# Controlled-long-tail T1 anchors

One-click, fail-closed launcher for the fixed full CONTROLLED T1 ANCHOR TRAINING RECIPE V1. It never runs T2–T6 and never touches historical replay workspaces.

In [ ]:
# 0 — Only edit this cell
ANCHOR_RECIPE = "fast"
CONDITIONS = ["lt100", "lt50", "lt10"]
SEED = 0
RUN_SMOKE_TEST = True
RUN_TRAINING = True
RUN_EVALUATION = True
TOTAL_GPU_BUDGET_HOURS = 14.0
FAST_DEFAULT_UPDATES = 12000
FINAL_EVALUATION_SECONDS = 1800
SAFETY_RESERVE_SECONDS = 1800
SETUP_SECONDS = 900
BENCHMARK_ITERATIONS = 20
DRIVE_ROOT = "/content/drive/MyDrive/OWL"

OWL_REPOSITORY = "https://github.com/gubiczam/owod-active.git"
OWL_COMMIT = "8a9c5a97d23f4532240d7be4852e3bc98dc2060b"
PROB_REPOSITORY = "https://github.com/gubiczam/PROB.git"
PROB_COMMIT = "4c66be1a52cad9360e09c729e9134aba8fe0b531"
EXPECTED_PYTHON = (3, 13)
EXPECTED_TORCH = "2.11.0+cu128"
EXPECTED_TORCHVISION = "0.26.0+cu128"
EXPECTED_CUDA = "12.8"
DINO_SHA256 = "156f8c4166a23dc2951ae811e39d76a06269c565932edf647c0187e65cd7aa7c"
assert ANCHOR_RECIPE == "fast"
assert CONDITIONS == ["lt100", "lt50", "lt10"]
assert 0 < TOTAL_GPU_BUDGET_HOURS <= 15.0
assert SEED == 0 and BENCHMARK_ITERATIONS == 20 and FAST_DEFAULT_UPDATES <= 17730

In [ ]:
# 1 — Mount Drive, pin repositories, and install the reviewed OWL package
import hashlib
import importlib
import importlib.metadata
import json
import subprocess
import sys
import time
from pathlib import Path

from google.colab import drive

assert sys.version_info[:2] == EXPECTED_PYTHON, sys.version
SESSION_STARTED_UNIX = time.time()
drive.mount("/content/drive", force_remount=False)
DRIVE = Path(DRIVE_ROOT)
DRIVE.mkdir(parents=True, exist_ok=True)
probe = DRIVE / ".controlled_lt_write_probe"
probe.write_text("ok")
assert probe.read_text() == "ok"
probe.unlink()


def checked(command, **kwargs):
    print("+", " ".join(map(str, command)))
    return subprocess.run(command, check=True, text=True, **kwargs)


def capture(command, **kwargs):
    return checked(command, capture_output=True, **kwargs).stdout.strip()


def pinned_checkout(path, repository, commit):
    path = Path(path)
    if not path.exists():
        checked(["git", "clone", "--filter=blob:none", "--no-checkout", repository, str(path)])
    assert (path / ".git").is_dir()
    actual_origin = capture(["git", "remote", "get-url", "origin"], cwd=path).removesuffix(".git")
    expected_origin = repository.removesuffix(".git")
    assert actual_origin == expected_origin, (actual_origin, expected_origin)
    checked(["git", "fetch", "--depth", "1", "origin", commit], cwd=path)
    checked(["git", "reset", "--hard", commit], cwd=path)
    checked(["git", "clean", "-fdx"], cwd=path)
    assert capture(["git", "rev-parse", "HEAD"], cwd=path) == commit
    return path


ROOT = pinned_checkout("/content/owod-active", OWL_REPOSITORY, OWL_COMMIT)
checked(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "-q",
        "-e",
        f"{ROOT}[plots]",
    ]
)
sys.path.insert(0, str(ROOT))
from owl import t1_anchor_fast

assert t1_anchor_fast.FAST_RECIPE_VERSION == "controlled_t1_anchor_fast_v1"

print("OWL PIN PASS:", OWL_COMMIT)

In [ ]:
# 2 — Reuse the proven Python-3.13 PROB dependency and compiled-MSDA bootstrap
PROB = pinned_checkout("/content/PROB", PROB_REPOSITORY, PROB_COMMIT)


def version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def available(name):
    return (
        subprocess.run(
            [sys.executable, "-c", f"import {name}"], cwd=PROB, capture_output=True, check=False
        ).returncode
        == 0
    )


if version("einops") != "0.5.0" or not available("einops"):
    checked([sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", "einops==0.5.0"])
if version("pycocotools") != "2.0.5" or not available("pycocotools"):
    checked([sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", "Cython==3.1.3"])
    checked(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--no-build-isolation",
            "--no-deps",
            "--force-reinstall",
            "pycocotools==2.0.5",
        ]
    )
wheels = {
    "pandas": "pandas==2.3.2",
    "seaborn": "seaborn==0.13.2",
    "tqdm": "tqdm==4.67.1",
    "wandb": "wandb==0.18.7",
}
missing = [spec for module, spec in wheels.items() if not available(module)]
if missing:
    checked([sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", *missing])
pip_check = subprocess.run(
    [sys.executable, "-m", "pip", "check"], text=True, capture_output=True, check=False
)
if pip_check.returncode and "jedi" in (pip_check.stdout + pip_check.stderr).lower():
    checked([sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", "jedi==0.19.2"])
    pip_check = subprocess.run(
        [sys.executable, "-m", "pip", "check"], text=True, capture_output=True, check=False
    )
if pip_check.returncode:
    print(pip_check.stdout + pip_check.stderr)
    raise RuntimeError("pip check failed")

DINO = PROB / "models/dino_resnet50_pretrain.pth"
if not DINO.is_file():
    partial = DINO.with_suffix(".pth.part")
    if partial.exists():
        partial.unlink()
    checked(
        [
            "curl",
            "--fail",
            "--location",
            "--retry",
            "3",
            "--output",
            str(partial),
            "https://dl.fbaipublicfiles.com/dino/dino_resnet50_pretrain/dino_resnet50_pretrain.pth",
        ]
    )
    assert hashlib.sha256(partial.read_bytes()).hexdigest() == DINO_SHA256
    partial.replace(DINO)
assert hashlib.sha256(DINO.read_bytes()).hexdigest() == DINO_SHA256


def msda_probe():
    code = """import json,torch
from models.ops.functions import ms_deform_attn_func as w
from models.ops.modules import ms_deform_attn as d
print(json.dumps({"cuda":torch.cuda.is_available(),"gpu":torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,"torch":torch.__version__,"torchvision":__import__("torchvision").__version__,"cuda_version":torch.version.cuda,"wrapper":bool(w.MSDA_AVAILABLE),"downstream":bool(d.MSDA_AVAILABLE),"path":getattr(w.MSDA,"__file__",None)}))"""
    result = subprocess.run(
        [sys.executable, "-c", code], cwd=PROB, text=True, capture_output=True, check=False
    )
    return (
        json.loads(result.stdout.splitlines()[-1])
        if result.returncode == 0 and result.stdout
        else {}
    )


MSDA = msda_probe()
if not (MSDA.get("wrapper") and MSDA.get("downstream")):
    checked([sys.executable, "-m", "pip", "install", "-q", "ninja"])
    checked(
        [sys.executable, "-m", "pip", "install", "--no-build-isolation", "."],
        cwd=PROB / "models/ops",
    )
    MSDA = msda_probe()
assert MSDA.get("cuda") and MSDA.get("wrapper") and MSDA.get("downstream"), MSDA
assert (
    MSDA["torch"] == EXPECTED_TORCH
    and MSDA["torchvision"] == EXPECTED_TORCHVISION
    and MSDA["cuda_version"] == EXPECTED_CUDA
), MSDA
assert "T4" in MSDA["gpu"], MSDA

coco_smoke = r"""import numpy as np, torch
if "float" not in np.__dict__: np.float=float
if "NPY_OWNDATA" not in np.__dict__: np.NPY_OWNDATA=4
from pycocotools.coco import COCO
from datasets.coco_eval import CocoEvaluator
c=COCO(); c.dataset={"info":{},"licenses":[],"images":[{"id":1,"width":32,"height":32}],"categories":[{"id":1,"name":"x","supercategory":"x"}],"annotations":[{"id":1,"image_id":1,"category_id":1,"bbox":[4.,5.,10.,11.],"area":110.,"iscrowd":0}]}; c.createIndex(); e=CocoEvaluator(c,("bbox",)); e.update({1:{"boxes":torch.tensor([[4.,5.,14.,16.]]),"scores":torch.tensor([.99]),"labels":torch.tensor([1])}}); e.synchronize_between_processes(); e.accumulate(); assert float(e.coco_eval["bbox"].stats[0])>.99"""
coco_smoke = coco_smoke.replace("e.accumulate(); assert", "e.accumulate(); e.summarize(); assert")
checked([sys.executable, "-c", coco_smoke], cwd=PROB)
print("PROB CUDA/MSDA/COCO BOOTSTRAP PASS:", MSDA)

In [ ]:
# 3 — Materialize the requested JPEG union and reuse one frozen initialization
WORK_ROOT = DRIVE / "anchors/controlled_lt_fast_v1/seed0"
WORK_ROOT.mkdir(parents=True, exist_ok=True)
JPEG_ROOT = Path("/content/data/controlled_lt/JPEGImages")
LEGACY_SHARED_INIT = DRIVE / "anchors/controlled_lt_v1/seed0/prob_t1_seed0_init.pth"
INIT = LEGACY_SHARED_INIT
assert INIT.is_file(), f"Frozen controlled-anchor initialization is missing: {INIT}"
checked(
    [
        sys.executable,
        str(ROOT / "tools/materialize_t1_anchor_images.py"),
        "--conditions",
        ",".join(CONDITIONS),
        "--jpeg-root",
        str(JPEG_ROOT),
        "--execute",
    ]
)
INIT_SHA = hashlib.sha256(INIT.read_bytes()).hexdigest()
INIT_META = json.loads(INIT.with_suffix(".initialization.json").read_text())
assert INIT_META["sha256"] == INIT_SHA and INIT_META["prob_commit"] == PROB_COMMIT
assert (
    INIT_META["torch_version"] == EXPECTED_TORCH
    and INIT_META["torchvision_version"] == EXPECTED_TORCHVISION
    and INIT_META["cuda_version"] == EXPECTED_CUDA
)
print("SHARED INITIALIZATION PASS:", INIT, INIT_SHA, INIT_META["model_state_sha256"])

In [ ]:
# 4 — Deterministically materialize/verify each FAST-isolated filtered-XML view
WORKSPACES = {condition: WORK_ROOT / f"t1_anchor_fast__{condition}__seed0" for condition in CONDITIONS}
for condition, workspace in WORKSPACES.items():
    state = t1_anchor_fast.workspace_state(workspace, condition)
    print(condition, "initial state:", state)
    if state == "DONE":
        continue
    if state == "FAILED":
        raise RuntimeError(f"{condition}: inspect failed FAST workspace {workspace}")
    data_root = Path(f"/content/data/controlled_lt_fast/{condition}/OWOD")
    checked(
        [
            sys.executable,
            str(ROOT / "tools/prepare_t1_anchor_fast.py"),
            "--condition",
            condition,
            "--prob-root",
            str(PROB),
            "--work-root",
            str(WORK_ROOT),
            "--data-root",
            str(data_root),
            "--jpeg-root",
            str(JPEG_ROOT),
            "--initialization",
            str(INIT),
            "--initialization-sha",
            INIT_SHA,
            "--owl-commit",
            OWL_COMMIT,
        ]
    )
print("CONTROLLED T1 ANCHOR FAST DATA PREFLIGHTS PASS")

In [ ]:
# 5 — Exact FAST path: 5 warmup + 20 measured real CUDA optimizer updates
SMOKES = {}
for condition, workspace in WORKSPACES.items():
    receipt = workspace / "cuda_benchmark_fast_v1.json"
    if RUN_SMOKE_TEST and not receipt.is_file():
        checked(
            [
                sys.executable,
                str(ROOT / "tools/train_t1_anchor_fast.py"),
                "--condition",
                condition,
                "--prob-root",
                str(PROB),
                "--workspace",
                str(workspace),
                "--initialization",
                str(INIT),
                "--initialization-sha",
                INIT_SHA,
                "--owl-commit",
                OWL_COMMIT,
                "--benchmark-iterations",
                str(BENCHMARK_ITERATIONS),
                "--benchmark",
                "--execute",
            ]
        )
    if receipt.is_file():
        SMOKES[condition] = json.loads(receipt.read_text())
    else:
        raise RuntimeError(f"{condition}: FAST planning forbidden without benchmark receipt")
assert set(SMOKES) == set(CONDITIONS)
print("FAST MEASURED SEC/UPDATE:", {c: s["seconds_per_optimizer_update"] for c, s in SMOKES.items()})
print("FAST PEAK GPU MEMORY GiB:", {c: s["peak_gpu_memory_bytes"] / (1 << 30) for c, s in SMOKES.items()})

In [ ]:
# 6 — Freeze one equal FAST budget from timing only; no AP is available or read here
PLAN_PATH = WORK_ROOT / "fast_recipe_plan.json"
checked(
    [
        sys.executable, str(ROOT / "tools/plan_t1_anchor_fast.py"),
        "--work-root", str(WORK_ROOT), "--output", str(PLAN_PATH),
        "--total-gpu-budget-hours", str(TOTAL_GPU_BUDGET_HOURS),
        "--default-updates", str(FAST_DEFAULT_UPDATES),
        "--final-evaluation-seconds", str(FINAL_EVALUATION_SECONDS),
        "--safety-reserve-seconds", str(SAFETY_RESERVE_SECONDS),
        "--setup-seconds", str(SETUP_SECONDS),
    ]
)
FAST_PLAN = json.loads(PLAN_PATH.read_text())
t1_anchor_fast.validate_plan(FAST_PLAN)
print("FAST RECIPE PLAN")
print("Measured T4 sec/update:", FAST_PLAN["seconds_per_update"])
print("Frozen optimizer updates/condition:", FAST_PLAN["frozen_optimizer_updates_per_condition"])
print("Training estimate/condition hours:", FAST_PLAN["training_seconds_per_condition"] / 3600)
print("Estimated final evaluation/condition hours:", FAST_PLAN["final_evaluation_seconds_per_condition"] / 3600)
print("Estimated total experiment hours:", FAST_PLAN["projected_total_seconds"] / 3600)
print("Total available hours:", TOTAL_GPU_BUDGET_HOURS)
print("Safety reserve hours:", FAST_PLAN["safety_reserve_seconds"] / 3600)
print("Latest safe expected completion UTC:", FAST_PLAN["latest_safe_expected_completion_utc"])
print("Decision:", FAST_PLAN["decision"])
assert FAST_PLAN["decision"] == "GO", "FAST plan does not fit safely; no training started"
SESSION_STOP_AT_UNIX = SESSION_STARTED_UNIX + TOTAL_GPU_BUDGET_HOURS * 3600

In [ ]:
# 7 — LT100 → LT50 → LT10, only while every unfinished condition still fits
def recorded_global_step(workspace):
    progress = workspace / "train/resume_progress.json"
    receipt = workspace / "training_session_fast_v1.json"
    source = progress if progress.is_file() else receipt
    return int(json.loads(source.read_text())["global_step"]) if source.is_file() else 0

if RUN_TRAINING or RUN_EVALUATION:
    for index, (condition, workspace) in enumerate(WORKSPACES.items()):
        state = t1_anchor_fast.workspace_state(workspace, condition)
        if state == "DONE":
            continue
        if state not in ("READY", "INCOMPLETE_RESUMABLE", "TRAINED_PENDING_EVAL"):
            raise RuntimeError(f"{condition}: {state}")
        required_seconds = FAST_PLAN["safety_reserve_seconds"]
        for future in CONDITIONS[index:]:
            future_state = t1_anchor_fast.workspace_state(WORKSPACES[future], future)
            if future_state == "DONE":
                continue
            if future_state != "TRAINED_PENDING_EVAL":
                remaining_steps = FAST_PLAN["frozen_optimizer_updates_per_condition"] - recorded_global_step(WORKSPACES[future])
                required_seconds += remaining_steps * FAST_PLAN["planning_seconds_per_update"]
            required_seconds += FAST_PLAN["final_evaluation_seconds_per_condition"]
        available_seconds = SESSION_STOP_AT_UNIX - time.time()
        print(condition, "state", state, "available h", available_seconds / 3600, "all-remaining required h", required_seconds / 3600)
        if state in ("READY", "INCOMPLETE_RESUMABLE") and available_seconds < required_seconds:
            print("FAST BUDGET STOP: not starting", condition, "because all remaining complete conditions no longer fit")
            break
        final_checkpoint = workspace / f"t1_fast_{condition}.pth"
        if RUN_TRAINING and not final_checkpoint.is_file():
            command = [
                sys.executable,
                str(ROOT / "tools/train_t1_anchor_fast.py"),
                "--condition",
                condition,
                "--prob-root",
                str(PROB),
                "--workspace",
                str(workspace),
                "--initialization",
                str(INIT),
                "--initialization-sha",
                INIT_SHA,
                "--owl-commit",
                OWL_COMMIT,
                "--plan",
                str(PLAN_PATH),
                "--stop-at-unix",
                str(SESSION_STOP_AT_UNIX),
                "--execute",
            ]
            checked(command)
        if RUN_EVALUATION and final_checkpoint.is_file():
            checked(
                [
                    sys.executable,
                    str(ROOT / "tools/evaluate_t1_anchor_fast.py"),
                    "--condition",
                    condition,
                    "--prob-root",
                    str(PROB),
                    "--workspace",
                    str(workspace),
                    "--execute",
                ]
            )
        state = t1_anchor_fast.workspace_state(workspace, condition)
        print(condition, "end state:", state)
        if state != "DONE":
            print(condition, "is not final; no unequal-budget comparison will be generated")
            break

In [ ]:
# 8 — FAST comparison only after all three exact-budget anchors are DONE
STATES = {
    condition: t1_anchor_fast.workspace_state(workspace, condition)
    for condition, workspace in WORKSPACES.items()
}
print("FINAL STATES:", STATES)
if all(value == "DONE" for value in STATES.values()):
    comparison = WORK_ROOT / "controlled_lt_fast_v1_comparison"
    if not (comparison / "summary.json").is_file():
        checked(
            [
                sys.executable,
                str(ROOT / "tools/compare_t1_anchors_fast.py"),
                "--work-root",
                str(WORK_ROOT),
                "--output",
                str(comparison),
            ]
        )
    summary = json.loads((comparison / "summary.json").read_text())
    assert summary["recipe_version"] == "controlled_t1_anchor_fast_v1"
    assert summary["conditions"] == ["lt10", "lt50", "lt100"]
    print("CONTROLLED T1 ANCHOR FAST EXPERIMENT COMPLETE", comparison)
else:
    print(
        "CONTROLLED T1 ANCHOR FAST INCOMPLETE — comparison withheld until LT10/LT50/LT100 are all DONE"
    )